# Part 1A — FDIC Acquisition and Source Verification

**BFT65B | A Big-Data Pipeline for Bank Failure Early Warning**

**Objective.** Retrieve the complete FDIC Financials panel for 1984Q1–2026Q2, together with the complete Failures and Institutions tables. Verify all record counts, typed schemas, source keys and SHA-256 identities before releasing the source contract.

Run cells **in sequence** on a Google Colab CPU runtime. Original FDIC data and verification files belong on Drive; only executable code and compact notebook outputs belong on GitHub.

## 1. Repository and pinned runtime

Clone the exact repository state used for the experiment. The Git SHA is retained for later reporting.

In [1]:
import subprocess, sys, tempfile
from pathlib import Path

REPOSITORY = 'https://github.com/NguyenDuong0508/11230945_BFT65B_Final_Research_BigData.git'
REPO = Path(tempfile.mkdtemp(prefix="bft65b_p1a_")) / "repository"
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(REPO)], check=True)
COMMIT_SHA = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
print("Repository:", REPOSITORY)
print("Git revision:", COMMIT_SHA)

Repository: https://github.com/NguyenDuong0508/11230945_BFT65B_Final_Research_BigData.git
Git revision: e12f8e16e8d3bbb06455aced70689a819767cdaf


## 2. Run unit and integration tests

Integration tests use synthetic FDIC responses and local Arrow files. This gate must pass without errors or skips before accessing the live API.

In [2]:
test = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-rs", "tests"],
    cwd=REPO, text=True, capture_output=True
)
print(test.stdout)
if test.stderr:
    print(test.stderr)
assert test.returncode == 0, "Part 1 test suite failed"
assert " skipped" not in test.stdout and "SKIPPED" not in test.stdout, "A required integration test was skipped"
print("Part 1 unit/integration tests: PASSED")

..........                                                               [100%]
10 passed in 1.30s

Part 1 unit/integration tests: PASSED


## 3. Mount Drive and define a stable acquisition path

The same `DRIVE_ROOT` is used by Notebook 02. The output path is fixed, while each checkpoint is content-addressed and verified.

In [3]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/BFT65B_Final_Research")
SOURCE_DIR = DRIVE_ROOT / "part1_data"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Acquisition directory:", SOURCE_DIR)

Mounted at /content/drive
Acquisition directory: /content/drive/MyDrive/BFT65B_Final_Research/part1_data


## 4. Download and verify the complete FDIC scope

Each quarterly request is paginated; completed Parquet partitions are skipped only if their hashes, schema, record counts, quarter values and keys still match the stored manifest. Historical failures without a certificate number are retained. No cleaning or labels are introduced here.

In [4]:
command = [sys.executable, str(REPO / "run_part1.py"), "acquire",
           "--out", str(SOURCE_DIR)]
subprocess.run(command, cwd=REPO, check=True)
print("Acquisition command completed")

Acquisition command completed


## 5. Independent full-scope verification

A valid result must cover exactly 170 quarters. Counts are taken from verified source files, never copied from a reference report.

In [5]:
import json
import sys
sys.path.insert(0, str(REPO / "src"))
from bft65b.part1.acquisition import verify_acquisition
from bft65b.part1.io_utils import read_json, sha256

VERIFICATION = verify_acquisition(SOURCE_DIR, require_full_scope=True)
CONTRACT = read_json(SOURCE_DIR / "acquisition_contract.json")
assert VERIFICATION["success"] is True
assert VERIFICATION["expected_quarters"] == VERIFICATION["verified_quarters"] == 170
assert VERIFICATION["missing_quarters"] == [] and VERIFICATION["invalid_artifacts"] == []
assert VERIFICATION["financial_rows"] > 0
assert VERIFICATION["failure_rows"] > 0 and VERIFICATION["institution_rows"] > 0
assert sha256(SOURCE_DIR / "manifest.json") == CONTRACT["manifest_sha256"]
assert sha256(SOURCE_DIR / "acquisition_verification.json") == CONTRACT["verification_sha256"]
print(json.dumps(VERIFICATION, indent=2))
print("Part 1A source verification: PASSED")

{
  "success": true,
  "manifest_sha256": "b7afe4c70e4726fd2b588f643d05ed103f5931f4bdc486b84cfd59723b6d45b5",
  "scope_start": "1984-03-31",
  "scope_end": "2026-06-30",
  "full_exam_scope": true,
  "expected_quarters": 170,
  "verified_quarters": 170,
  "missing_quarters": [],
  "invalid_artifacts": [],
  "financial_rows": 1682738,
  "distinct_financial_certs": 24064,
  "failure_rows": 4117,
  "failures_without_cert": 488,
  "institution_rows": 27838
}
Part 1A source verification: PASSED


## 6. Record the Part 2 handoff

`acquisition_contract.json` is the sole canonical contract for downstream source loading. Part 2 will re-validate the source and must not infer data quality or outcome labels from Part 1 completeness.

In [6]:
print("Source dataset:", SOURCE_DIR)
print("Financials:", f"{VERIFICATION['financial_rows']:,}")
print("Failures:", f"{VERIFICATION['failure_rows']:,}")
print("Institutions:", f"{VERIFICATION['institution_rows']:,}")
print("Failures without CERT:", VERIFICATION['failures_without_cert'])
print("Source manifest SHA-256:", CONTRACT['manifest_sha256'])
print("Contract:", SOURCE_DIR / "acquisition_contract.json")
print("Git revision:", COMMIT_SHA)

Source dataset: /content/drive/MyDrive/BFT65B_Final_Research/part1_data
Financials: 1,682,738
Failures: 4,117
Institutions: 27,838
Failures without CERT: 488
Source manifest SHA-256: b7afe4c70e4726fd2b588f643d05ed103f5931f4bdc486b84cfd59723b6d45b5
Contract: /content/drive/MyDrive/BFT65B_Final_Research/part1_data/acquisition_contract.json
Git revision: e12f8e16e8d3bbb06455aced70689a819767cdaf
